In [2]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

In [3]:
# Set cố định các random sssed để đảm bảo các phép toán ngẫu nhiên sẽ tạo ra kết quả giống nhau
np.random.seed(42)
tf.random.set_seed(42)

In [4]:
pos_df = pd.read_csv("./review-data/positive_data.csv")
pos_df.head(5)

,Rate,Review,Label
0,9.0,Khu ẩm thực với đa dạng đồ lại còn bày trí đẹ...,1
1,9.0,Lúc nào đến aeon là lúc đấy phải tống một đốn...,1
2,10.0,Bánh ngon lại rẻ chê đâu được gần hết các loạ...,1
3,9.0,Ngon rẻ,1
4,9.6,Tôi sắp chết vì ngập trong sushi mấttttt Lên ...,1


In [5]:
neg_df = pd.read_csv("./review-data/negative_data.csv")
neg_df.head(5)

,Rate,Review,Label
0,4.0,Mình thề là mình ko thể cảm nổi đồ ăn ở aeon ...,-1
1,3.8,Đôi khi thèm lên là bất chấp nắng nóng phi Và...,-1
2,3.8,Ngõ treo biển cafe trứng đúng kiểu phố cổ hà ...,-1
3,3.8,Mình thấy địa chỉ cafe Giảng ở Nguyễn Hữu Huâ...,-1
4,2.2,Mình là người Hà Nội và cũng cực kỳ khó tính ...,-1


Đếm số giá trị positive và negative

In [6]:
pos_df['Label'].value_counts()

Label
1    2642
Name: count, dtype: int64

In [7]:
neg_df['Label'].value_counts()

Label
-1    1765
Name: count, dtype: int64

In [8]:
pos_df_sampled = pos_df.sample(n=len(neg_df), random_state=42)


Kết hợp dữ liệu positive và negative vào cùng 1 DataFrame

In [9]:
df = pd.concat([pos_df_sampled, neg_df])
df['Label'].value_counts()

Label
 1    1765
-1    1765
Name: count, dtype: int64

Phân phối lại các sample positve và negative trong df.
Ở đây ra sẽ chọn frac=1 để chọn tất cả các sample trong df với thứ tự ngẫu nhiên

In [10]:
df = df.sample(frac = 1).reset_index(drop=True)
df.head(10)

,Rate,Review,Label
0,9.0,Nhà mình người đi ăn mà hóa đơn là đắt hay rẻ...,1
1,3.4,Mình nói thật mình ăn ở đây nhiều rồi nhưng h...,-1
2,2.8,Đặc sệt hơi hướng nhà Nhân viên thì đông như ...,-1
3,9.0,Đến lần k phải giờ cao điểm nên khá thoải mái...,1
4,9.4,Soya bean ăn chất lượng,1
5,4.0,Thấy được review hay ho nên đến Nhưng chả có ...,-1
6,2.2,Nhân viên nhiệt tình nhưng chưa nhanh phục vụ...,-1
7,9.4,Hôm nay ra Surf n fries có ưu đãi ăn khoai Mỗ...,1
8,9.6,Đây là quán quen của mình và Quán không gian ...,1
9,3.4,Nói chung là bánh cũng không phải quá đặc mà ...,-1


Đầu ra của mô hình phân loại 2 lớp là giá trị 0 hoặc 1.
Vì vậy cần chuyển các giá trị -1 thành 0 bằng DataFrame.replace

In [11]:
df['Label'] = df['Label'].replace({-1 : 0})
df['Label'].value_counts()

Label
1    1765
0    1765
Name: count, dtype: int64

Chia tập train - test

In [12]:
X = df['Review'].values
y = df['Label'].values
X.shape, y.shape

((3530,), (3530,))

Sử dụng hàm train_test_split để chia tập train và test theo tỉ lệ 8:2

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = np.array(X_train)
X_test = np.array(X_test)
y_train = np.array(y_train)
y_test = np.array(y_test)
X_train.shape, X_test.shape

((2824,), (706,))

Tokenize dữ liệu train - test

In [14]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences

In [15]:
# Chọn kính thức từ điển là 10000
# Độ dài lớn nhất của 1 bình luận là 400 tokens
vocab_size = 10000
max_length = 400

# Tạo Tokenizer và fit trên tập train
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

# Sử dụng tokenizer đã fit để tokenize các bình luận trên tập train
X_train_seqs = tokenizer.texts_to_sequences(X_train)
X_train_padded = pad_sequences(X_train_seqs, maxlen=max_length, padding="post", truncating="post")

# Sử dụng tokenizer để fit để tokenize các bình luận trên tập test
X_test_seqs = tokenizer.texts_to_sequences(X_test)
X_test_padded = pad_sequences(X_test_seqs, maxlen=max_length, padding="post", truncating="post")

print("Độ dài của train dataset:", len(X_train_padded))
print("Độ dài của test dataset", len(X_test_padded))

Độ dài của train dataset: 2824
Độ dài của test dataset 706
